In [1]:
# CÉLULA 1 — Carga do dataframe processado

import os
import pandas as pd
import numpy as np
import plotly.express as px

BASE_DIR = r"C:\Users\Usuario\Downloads\Bruno_USP\Documentos\Censo_Religião"
caminho_dados = os.path.join(BASE_DIR, "data", "processed", "censo_religiao_renda.csv")

df = pd.read_csv(caminho_dados)

print(f"Registros: {len(df)}")
print(f"Colunas: {df.columns.tolist()}")
df.head()

Registros: 410000
Colunas: ['ano', 'uf', 'sexo', 'idade', 'raca', 'escolaridade_original', 'renda', 'urbano_rural', 'grupo_religioso', 'renda_sm']


,ano,uf,sexo,idade,raca,escolaridade_original,renda,urbano_rural,grupo_religioso,renda_sm
0,1991,50,1,34.0,1,10.0,450000.0,1,Outras,10.714286
1,1991,50,2,20.0,1,10.0,80000.0,1,Sem religião,1.904762
2,1991,50,2,0.0,1,NaN,NaN,1,Sem religião,NaN
3,1991,50,1,47.0,1,17.0,400000.0,1,Mediúnica/Espírita,9.523810
4,1991,50,2,42.0,1,11.0,200000.0,1,Mediúnica/Espírita,4.761905


In [2]:
# CÉLULA 2 — Criando a variável região e rótulos legíveis

# O primeiro dígito do código da UF indica a região
def definir_regiao(uf):
    primeiro = str(uf)[0]
    if primeiro == "1":
        return "Norte"
    elif primeiro == "2":
        return "Nordeste"
    elif primeiro == "3":
        return "Sudeste"
    elif primeiro == "4":
        return "Sul"
    elif primeiro == "5":
        return "Centro-Oeste"
    return "Indefinida"

df["regiao"] = df["uf"].apply(definir_regiao)

# Rótulos de sexo e raça
df["sexo_label"] = df["sexo"].map({1: "Masculino", 2: "Feminino"})
df["raca_label"] = df["raca"].map({1: "Branca", 2: "Preta", 3: "Amarela", 4: "Parda", 5: "Indígena"})

print(df["regiao"].value_counts())
print()
print(df["raca_label"].value_counts())

regiao
Nordeste        135000
Norte           105000
Sudeste          65000
Centro-Oeste     60000
Sul              45000
Name: count, dtype: int64

raca_label
Parda       208233
Branca      171232
Preta        19314
Indígena      7617
Amarela       2258
Name: count, dtype: int64


In [3]:
# CÉLULA 3 — Limpeza analítica

print(f"Antes da limpeza: {len(df)} registros")

# 1. Apenas idade produtiva ampliada (15 a 70 anos)
df_limpo = df[(df["idade"] >= 15) & (df["idade"] <= 70)].copy()
print(f"Após filtro de idade (15-70): {len(df_limpo)}")

# 2. Renda válida (não nula e não negativa)
df_limpo = df_limpo[df_limpo["renda_sm"].notna()]
df_limpo = df_limpo[df_limpo["renda_sm"] >= 0]
print(f"Após filtro de renda válida: {len(df_limpo)}")

# 3. Religião classificada (remove nulos e não classificados)
df_limpo = df_limpo[df_limpo["grupo_religioso"].notna()]
df_limpo = df_limpo[df_limpo["grupo_religioso"] != "Não classificado"]
print(f"Após filtro de religião válida: {len(df_limpo)}")

# 4. Remove outliers extremos de renda (acima de 200 salários mínimos)
df_limpo = df_limpo[df_limpo["renda_sm"] <= 200]
print(f"Após remover rendas > 200 SM: {len(df_limpo)}")

Antes da limpeza: 410000 registros
Após filtro de idade (15-70): 259992
Após filtro de renda válida: 258999
Após filtro de religião válida: 258890
Após remover rendas > 200 SM: 258861


In [4]:
print(df.groupby("ano")["renda_sm"].median())


ano
1991    0.00000
2000    0.53000
2010    0.58824
Name: renda_sm, dtype: float64


In [5]:
# CÉLULA 4 — Filtro de renda positiva e base analítica final

df_renda = df_limpo[df_limpo["renda_sm"] > 0].copy()

print(f"Base com renda positiva: {len(df_renda)} registros")
print()
print("Mediana de renda (SM) por ano:")
print(df_renda.groupby("ano")["renda_sm"].median())
print()
print("Registros por ano:")
print(df_renda["ano"].value_counts().sort_index())

Base com renda positiva: 162662 registros

Mediana de renda (SM) por ano:
ano
1991    0.952381
2000    1.660000
2010    1.000000
Name: renda_sm, dtype: float64

Registros por ano:
ano
1991    43784
2000    53243
2010    65635
Name: count, dtype: int64


In [6]:
# CÉLULA 5 — Renda por grupo religioso (média e mediana)

grupos_foco = ["Católica", "Evangélica Tradicional", "Evangélica Pentecostal",
               "Sem religião", "Mediúnica/Espírita"]

df_foco = df_renda[df_renda["grupo_religioso"].isin(grupos_foco)]

for ano in [1991, 2000, 2010]:
    print(f"\n========== ANO: {ano} ==========")
    dados_ano = df_foco[df_foco["ano"] == ano]
    
    contagem = dados_ano.groupby("grupo_religioso")["renda_sm"].count()
    media = dados_ano.groupby("grupo_religioso")["renda_sm"].mean().round(2)
    mediana = dados_ano.groupby("grupo_religioso")["renda_sm"].median().round(2)
    
    resumo = pd.DataFrame({"n": contagem, "media": media, "mediana": mediana})
    print(resumo.sort_values("mediana", ascending=False))


========== ANO: 1991 ==========
                            n  media  mediana
grupo_religioso                              
Mediúnica/Espírita        650   7.40     3.57
Sem religião             1859   3.80     1.19
Evangélica Tradicional   1511   2.77     1.14
Católica                37658   2.19     0.95
Evangélica Pentecostal    928   1.59     0.95

========== ANO: 2000 ==========
                            n  media  mediana
grupo_religioso                              
Mediúnica/Espírita        845  13.29     7.95
Evangélica Tradicional   2860   4.50     1.99
Sem religião             3238   6.06     1.99
Católica                40618   4.38     1.59
Evangélica Pentecostal   4834   2.86     1.46

========== ANO: 2010 ==========
                            n  media  mediana
grupo_religioso                              
Mediúnica/Espírita        886   8.10     3.92
Evangélica Tradicional   3235   2.61     1.25
Sem religião             4681   3.12     1.18
Católica                451

In [7]:
# CÉLULA 6 — Variável de superior completo (harmonizada entre censos)

def tem_superior(linha):
    if linha["ano"] == 1991:
        return 15 <= linha["escolaridade_original"] <= 17
    elif linha["ano"] == 2000:
        return 15 <= linha["escolaridade_original"] <= 17
    elif linha["ano"] == 2010:
        return linha["escolaridade_original"] == 4
    return False

df_renda["superior"] = df_renda.apply(tem_superior, axis=1)

print(df_renda.groupby("ano")["superior"].mean().round(3))

ano
1991    0.050
2000    0.086
2010    0.093
Name: superior, dtype: float64


In [8]:
# CÉLULA 7 — Coorte 1: Elite histórica
# Homens brancos, superior completo, nascidos entre 1956 e 1966
# (25-35 anos em 1991 → 34-44 em 2000 → 44-54 em 2010)

grupos_foco = ["Católica", "Evangélica Tradicional", "Evangélica Pentecostal",
               "Sem religião", "Mediúnica/Espírita"]

def filtrar_coorte(df, ano, idade_min, idade_max, sexo=None, racas=None, superior=None):
    filtro = (df["ano"] == ano) & (df["idade"] >= idade_min) & (df["idade"] <= idade_max)
    if sexo is not None:
        filtro = filtro & (df["sexo"] == sexo)
    if racas is not None:
        filtro = filtro & (df["raca"].isin(racas))
    if superior is not None:
        filtro = filtro & (df["superior"] == superior)
    return df[filtro & df["grupo_religioso"].isin(grupos_foco)]

# Elite: homens (1), brancos (1), com superior
c1_1991 = filtrar_coorte(df_renda, 1991, 25, 35, sexo=1, racas=[1], superior=True)
c1_2000 = filtrar_coorte(df_renda, 2000, 34, 44, sexo=1, racas=[1], superior=True)
c1_2010 = filtrar_coorte(df_renda, 2010, 44, 54, sexo=1, racas=[1], superior=True)

for nome, c in [("1991 (25-35)", c1_1991), ("2000 (34-44)", c1_2000), ("2010 (44-54)", c1_2010)]:
    print(f"\n===== Coorte 1 em {nome} | n = {len(c)} =====")
    mediana = c.groupby("grupo_religioso")["renda_sm"].median().round(2)
    media = c.groupby("grupo_religioso")["renda_sm"].mean().round(2)
    n = c.groupby("grupo_religioso")["renda_sm"].count()
    print(pd.DataFrame({"n": n, "media": media, "mediana": mediana}).sort_values("mediana", ascending=False))


===== Coorte 1 em 1991 (25-35) | n = 270 =====
                          n  media  mediana
grupo_religioso                            
Sem religião             31  18.41    13.34
Mediúnica/Espírita       20  17.19    12.50
Católica                206  10.70     8.33
Evangélica Tradicional   13  10.80     6.14

===== Coorte 1 em 2000 (34-44) | n = 619 =====
                          n  media  mediana
grupo_religioso                            
Sem religião             78  27.74    25.49
Católica                453  25.14    19.87
Mediúnica/Espírita       36  24.69    19.87
Evangélica Tradicional   34  22.86    13.25
Evangélica Pentecostal   18   8.93     6.95

===== Coorte 1 em 2010 (44-54) | n = 372 =====
                          n  media  mediana
grupo_religioso                            
Sem religião             30  24.83    22.65
Mediúnica/Espírita       35  16.71    14.51
Católica                268  15.63    11.76
Evangélica Pentecostal   22  14.92     8.71
Evangélica Tradicion

In [9]:
# CÉLULA 8 — Coorte 2: Base histórica
# Mulheres negras (pretas + pardas), sem superior, nascidas entre 1956 e 1966

c2_1991 = filtrar_coorte(df_renda, 1991, 25, 35, sexo=2, racas=[2, 4], superior=False)
c2_2000 = filtrar_coorte(df_renda, 2000, 34, 44, sexo=2, racas=[2, 4], superior=False)
c2_2010 = filtrar_coorte(df_renda, 2010, 44, 54, sexo=2, racas=[2, 4], superior=False)

for nome, c in [("1991 (25-35)", c2_1991), ("2000 (34-44)", c2_2000), ("2010 (44-54)", c2_2010)]:
    print(f"\n===== Coorte 2 em {nome} | n = {len(c)} =====")
    mediana = c.groupby("grupo_religioso")["renda_sm"].median().round(2)
    media = c.groupby("grupo_religioso")["renda_sm"].mean().round(2)
    n = c.groupby("grupo_religioso")["renda_sm"].count()
    print(pd.DataFrame({"n": n, "media": media, "mediana": mediana}).sort_values("mediana", ascending=False))


===== Coorte 2 em 1991 (25-35) | n = 1740 =====
                           n  media  mediana
grupo_religioso                             
Mediúnica/Espírita        16   4.35     3.57
Evangélica Tradicional    53   1.70     1.02
Sem religião              52   2.27     1.00
Evangélica Pentecostal    33   1.17     0.95
Católica                1586   1.26     0.71

===== Coorte 2 em 2000 (34-44) | n = 1908 =====
                           n  media  mediana
grupo_religioso                             
Mediúnica/Espírita        20   5.52     3.51
Evangélica Tradicional    82   2.43     1.66
Sem religião              83   2.58     1.42
Católica                1468   2.23     1.03
Evangélica Pentecostal   255   1.96     1.03

===== Coorte 2 em 2010 (44-54) | n = 2557 =====
                           n  media  mediana
grupo_religioso                             
Mediúnica/Espírita        24   4.09     1.39
Evangélica Tradicional    96   2.03     1.00
Sem religião             109   1.16     1.0

In [10]:
# CÉLULA 9 — Coortes 3 e 4: Religião de berço
# Nascidos 1976-1986: tinham 5-15 anos em 1991 (religião do domicílio)
# e 24-34 anos em 2010 (educação concluída + renda inicial)

grupos_foco = ["Católica", "Evangélica Tradicional", "Evangélica Pentecostal", "Sem religião"]

# Em 1991: composição religiosa das crianças (usa df original, sem filtro de idade)
criancas_1991 = df[(df["ano"] == 1991) &
                   (df["idade"] >= 5) & (df["idade"] <= 15) &
                   (df["grupo_religioso"].isin(grupos_foco))]

print("===== Religião de berço (5-15 anos em 1991) =====")
print(criancas_1991["grupo_religioso"].value_counts())
print()
print("Proporções (%):")
print((criancas_1991["grupo_religioso"].value_counts(normalize=True) * 100).round(1))

# Em 2010: mesma geração adulta (24-34 anos)
adultos_2010 = df_renda[(df_renda["ano"] == 2010) &
                        (df_renda["idade"] >= 24) & (df_renda["idade"] <= 34) &
                        (df_renda["grupo_religioso"].isin(grupos_foco))]

print("\n===== Mesma geração em 2010 (24-34 anos) =====")
print("\nComposição religiosa (%):")
print((adultos_2010["grupo_religioso"].value_counts(normalize=True) * 100).round(1))

print("\nTaxa de superior completo por religião e sexo (%):")
taxa = adultos_2010.groupby(["grupo_religioso", "sexo_label"])["superior"].mean() * 100
print(taxa.round(1))

print("\nMediana de renda (SM) por religião e sexo:")
renda_med = adultos_2010.groupby(["grupo_religioso", "sexo_label"])["renda_sm"].median()
print(renda_med.round(2))

===== Religião de berço (5-15 anos em 1991) =====
grupo_religioso
Católica                  34572
Evangélica Tradicional     1229
Sem religião               1189
Evangélica Pentecostal      984
Name: count, dtype: int64

Proporções (%):
grupo_religioso
Católica                  91.0
Evangélica Tradicional     3.2
Sem religião               3.1
Evangélica Pentecostal     2.6
Name: proportion, dtype: float64

===== Mesma geração em 2010 (24-34 anos) =====

Composição religiosa (%):
grupo_religioso
Católica                  71.7
Evangélica Pentecostal    13.8
Sem religião               9.2
Evangélica Tradicional     5.4
Name: proportion, dtype: float64

Taxa de superior completo por religião e sexo (%):
grupo_religioso         sexo_label
Católica                Feminino      14.0
                        Masculino      8.2
Evangélica Pentecostal  Feminino       7.6
                        Masculino      3.5
Evangélica Tradicional  Feminino      22.8
                        Masculino     14

In [11]:
# CÉLULA 10 — Renda por religião e nível de ensino (geração de berço, 2010)

print("===== Renda (SM): COM superior completo =====")
com_sup = adultos_2010[adultos_2010["superior"] == True]
n_com = com_sup.groupby("grupo_religioso")["renda_sm"].count()
media_com = com_sup.groupby("grupo_religioso")["renda_sm"].mean().round(2)
med_com = com_sup.groupby("grupo_religioso")["renda_sm"].median().round(2)
print(pd.DataFrame({"n": n_com, "media": media_com, "mediana": med_com}).sort_values("mediana", ascending=False))

print("\n===== Renda (SM): SEM superior completo =====")
sem_sup = adultos_2010[adultos_2010["superior"] == False]
n_sem = sem_sup.groupby("grupo_religioso")["renda_sm"].count()
media_sem = sem_sup.groupby("grupo_religioso")["renda_sm"].mean().round(2)
med_sem = sem_sup.groupby("grupo_religioso")["renda_sm"].median().round(2)
print(pd.DataFrame({"n": n_sem, "media": media_sem, "mediana": med_sem}).sort_values("mediana", ascending=False))

print("\n===== Prêmio do diploma por religião (razão com/sem, pela mediana) =====")
premio = (med_com / med_sem).round(2)
print(premio.sort_values(ascending=False))

===== Renda (SM): COM superior completo =====
                           n  media  mediana
grupo_religioso                             
Sem religião             236   9.20     6.37
Católica                1353   6.31     3.53
Evangélica Tradicional   172   5.83     3.14
Evangélica Pentecostal   133   4.28     2.51

===== Renda (SM): SEM superior completo =====
                            n  media  mediana
grupo_religioso                              
Evangélica Tradicional    753   1.65     1.09
Sem religião             1345   1.73     1.01
Católica                11001   1.33     1.00
Evangélica Pentecostal   2239   1.39     1.00

===== Prêmio do diploma por religião (razão com/sem, pela mediana) =====
grupo_religioso
Sem religião              6.31
Católica                  3.53
Evangélica Tradicional    2.88
Evangélica Pentecostal    2.51
Name: renda_sm, dtype: float64


In [12]:
# CÉLULA 11 — Renda por religião, ensino e gênero (geração de berço, 2010)

for sexo in ["Masculino", "Feminino"]:
    print(f"\n{'='*50}")
    print(f"  {sexo.upper()}")
    print('='*50)
    
    base_sexo = adultos_2010[adultos_2010["sexo_label"] == sexo]
    
    print("\n--- COM superior ---")
    com_s = base_sexo[base_sexo["superior"] == True]
    n1 = com_s.groupby("grupo_religioso")["renda_sm"].count()
    media1 = com_s.groupby("grupo_religioso")["renda_sm"].mean().round(2)
    m1 = com_s.groupby("grupo_religioso")["renda_sm"].median().round(2)
    print(pd.DataFrame({"n": n1, "media": media1, "mediana": m1}).sort_values("mediana", ascending=False))
    
    print("\n--- SEM superior ---")
    sem_s = base_sexo[base_sexo["superior"] == False]
    n2 = sem_s.groupby("grupo_religioso")["renda_sm"].count()
    media2 = sem_s.groupby("grupo_religioso")["renda_sm"].mean().round(2)
    m2 = sem_s.groupby("grupo_religioso")["renda_sm"].median().round(2)
    print(pd.DataFrame({"n": n2, "media": media2, "mediana": m2}).sort_values("mediana", ascending=False))


  MASCULINO

--- COM superior ---
                          n  media  mediana
grupo_religioso                            
Sem religião            132  10.38     7.84
Evangélica Tradicional   62   7.30     5.86
Católica                536   8.30     4.90
Evangélica Pentecostal   40   7.17     3.57

--- SEM superior ---
                           n  media  mediana
grupo_religioso                             
Evangélica Tradicional   381   2.12     1.57
Evangélica Pentecostal  1105   1.83     1.37
Sem religião             843   1.91     1.22
Católica                5984   1.64     1.14

  FEMININO

--- COM superior ---
                          n  media  mediana
grupo_religioso                            
Sem religião            104   7.70     4.95
Católica                817   5.00     2.94
Evangélica Tradicional  110   5.01     2.71
Evangélica Pentecostal   93   3.04     1.96

--- SEM superior ---
                           n  media  mediana
grupo_religioso                             

In [13]:
# CÉLULA 12 — Taxa de diploma por religião, controlando demografia
# Geração nascida 1976-86, observada em 2010 (24-34 anos)

# Usamos df_limpo (não df_renda) para incluir quem não tem renda:
# a pergunta aqui é sobre educação, não renda
base_geracao = df_limpo[(df_limpo["ano"] == 2010) &
                        (df_limpo["idade"] >= 24) & (df_limpo["idade"] <= 34) &
                        (df_limpo["grupo_religioso"].isin(grupos_foco))].copy()

base_geracao["superior"] = base_geracao.apply(tem_superior, axis=1)

print("===== Taxa de superior (%) por religião — GERAL =====")
geral = base_geracao.groupby("grupo_religioso")["superior"].mean() * 100
n_geral = base_geracao.groupby("grupo_religioso")["superior"].count()
print(pd.DataFrame({"n": n_geral, "taxa_%": geral.round(1)}).sort_values("taxa_%", ascending=False))

print("\n===== Controlando por raça =====")
for raca in ["Branca", "Parda", "Preta"]:
    sub = base_geracao[base_geracao["raca_label"] == raca]
    taxa = sub.groupby("grupo_religioso")["superior"].mean() * 100
    n = sub.groupby("grupo_religioso")["superior"].count()
    tabela = pd.DataFrame({"n": n, "taxa_%": taxa.round(1)}).sort_values("taxa_%", ascending=False)
    print(f"\n--- {raca} ---")
    print(tabela[tabela["n"] >= 30])

print("\n===== Controlando por região =====")
for reg in ["Sudeste", "Nordeste", "Sul"]:
    sub = base_geracao[base_geracao["regiao"] == reg]
    taxa = sub.groupby("grupo_religioso")["superior"].mean() * 100
    n = sub.groupby("grupo_religioso")["superior"].count()
    tabela = pd.DataFrame({"n": n, "taxa_%": taxa.round(1)}).sort_values("taxa_%", ascending=False)
    print(f"\n--- {reg} ---")
    print(tabela[tabela["n"] >= 30])

===== Taxa de superior (%) por religião — GERAL =====
                            n  taxa_%
grupo_religioso                      
Evangélica Tradicional   1145    16.5
Sem religião             2111    12.8
Católica                16693     9.2
Evangélica Pentecostal   3246     4.6

===== Controlando por raça =====

--- Branca ---
                           n  taxa_%
grupo_religioso                     
Sem religião             733    23.9
Evangélica Tradicional   628    19.1
Católica                6484    15.5
Evangélica Pentecostal  1132     6.6

--- Parda ---
                           n  taxa_%
grupo_religioso                     
Evangélica Tradicional   412    15.3
Sem religião            1130     7.3
Católica                8782     5.1
Evangélica Pentecostal  1817     3.6

--- Preta ---
                          n  taxa_%
grupo_religioso                    
Evangélica Tradicional   57     7.0
Católica                993     5.1
Sem religião            152     3.3
Evangélica Pen

In [18]:
# CÉLULA 13 — Trajetória da coorte 1956-66 com filtros interativos

from ipywidgets import interact, Dropdown

janelas = {1991: (25, 35), 2000: (34, 44), 2010: (44, 54)}

cores = {"Católica": "#4169E1", "Evangélica Tradicional": "#2E8B57",
         "Evangélica Pentecostal": "#DC143C", "Sem religião": "#FF8C00",
         "Mediúnica/Espírita": "#9370DB"}

def grafico_coorte(genero, escolaridade, regiao, etnia):
    base_g = df_renda[df_renda["grupo_religioso"].isin(grupos_foco)]
    if genero != "Todos":
        base_g = base_g[base_g["sexo_label"] == genero]
    if escolaridade == "Com superior":
        base_g = base_g[base_g["superior"] == True]
    elif escolaridade == "Sem superior":
        base_g = base_g[base_g["superior"] == False]
    if regiao != "Brasil":
        base_g = base_g[base_g["regiao"] == regiao]
    if etnia == "Negra (preta + parda)":
        base_g = base_g[base_g["raca_label"].isin(["Preta", "Parda"])]
    elif etnia != "Todas":
        base_g = base_g[base_g["raca_label"] == etnia]

    linhas = []
    for ano, (i_min, i_max) in janelas.items():
        recorte = base_g[(base_g["ano"] == ano) &
                         (base_g["idade"] >= i_min) & (base_g["idade"] <= i_max)]
        med = recorte.groupby("grupo_religioso")["renda_sm"].median()
        n = recorte.groupby("grupo_religioso")["renda_sm"].count()
        for g in grupos_foco:
            if g in med.index and n[g] >= 10:
                linhas.append({"ano": ano, "grupo_religioso": g,
                               "renda_mediana": round(med[g], 2), "n": int(n[g])})
    df_g = pd.DataFrame(linhas)
    if len(df_g) == 0:
        print("Sem dados suficientes para esse recorte (n < 10 em todos os grupos).")
        return
    fig = px.line(df_g, x="ano", y="renda_mediana", color="grupo_religioso",
                  markers=True, custom_data=["n"],
                  color_discrete_map=cores,
                  title=f"Renda da geração 1956-66 — {genero}, {escolaridade}, {regiao}, {etnia}",
                  labels={"ano": "Censo", "renda_mediana": "Renda mediana (SM)",
                          "grupo_religioso": "Religião"})
    fig.update_traces(
        hovertemplate="%{y} SM (n=%{customdata[0]})"
    )
    fig.update_layout(xaxis=dict(tickvals=[1991, 2000, 2010]),
                      height=500, template="plotly_white",
                      hovermode="x unified",
                      hoverlabel=dict(bgcolor="white", font_color="black"))
    fig.show()

interact(grafico_coorte,
         genero=Dropdown(options=["Todos", "Masculino", "Feminino"], description="Gênero:"),
         escolaridade=Dropdown(options=["Todos", "Com superior", "Sem superior"], description="Ensino:"),
         regiao=Dropdown(options=["Brasil", "Norte", "Nordeste", "Sudeste", "Sul", "Centro-Oeste"],
                         description="Região:"),
         etnia=Dropdown(options=["Todas", "Branca", "Negra (preta + parda)", "Preta", "Parda",
                                 "Amarela", "Indígena"], description="Etnia:"));

interactive(children=(Dropdown(description='Gênero:', options=('Todos', 'Masculino', 'Feminino'), value='Todos…

In [19]:
# CÉLULA 14 — Geração 1976-86 em 2010: renda por religião e escolaridade + % diploma

from plotly.subplots import make_subplots
import plotly.graph_objects as go

def grafico_berco(genero, regiao, etnia):
    base_g = adultos_2010[adultos_2010["grupo_religioso"].isin(grupos_foco)].copy()

    if genero != "Todos":
        base_g = base_g[base_g["sexo_label"] == genero]
    if regiao != "Brasil":
        base_g = base_g[base_g["regiao"] == regiao]
    if etnia == "Negra (preta + parda)":
        base_g = base_g[base_g["raca_label"].isin(["Preta", "Parda"])]
    elif etnia != "Todas":
        base_g = base_g[base_g["raca_label"] == etnia]

    linhas = []
    for g in grupos_foco:
        sub_g = base_g[base_g["grupo_religioso"] == g]
        if len(sub_g) < 10:
            continue
        taxa = sub_g["superior"].mean() * 100
        for sup, rotulo in [(False, "Sem superior"), (True, "Com superior")]:
            sub = sub_g[sub_g["superior"] == sup]
            if len(sub) >= 10:
                linhas.append({"religiao": g, "escolaridade": rotulo,
                               "mediana": round(sub["renda_sm"].median(), 2),
                               "n": len(sub), "taxa_diploma": round(taxa, 1)})

    df_g = pd.DataFrame(linhas)
    if len(df_g) == 0:
        print("Sem dados suficientes para esse recorte.")
        return

    ordem = df_g[df_g["escolaridade"] == "Com superior"].sort_values(
        "mediana", ascending=False)["religiao"].tolist()
    # Religiões que só têm "Sem superior" no recorte entram no fim da ordem
    for r in df_g["religiao"].unique():
        if r not in ordem:
            ordem.append(r)

    fig = make_subplots(specs=[[{"secondary_y": True}]])

    cores_esc = {"Sem superior": "#A7C7E7", "Com superior": "#4169E1"}

    for esc in ["Sem superior", "Com superior"]:
        sub = df_g[df_g["escolaridade"] == esc].set_index("religiao").reindex(ordem).reset_index()
        fig.add_trace(go.Bar(x=sub["religiao"], y=sub["mediana"],
                             name=f"Mediana — {esc}",
                             marker_color=cores_esc[esc],
                             text=sub["mediana"],
                             customdata=sub[["n"]],
                             hovertemplate="%{y} SM (n=%{customdata[0]})"),
                      secondary_y=False)

    sub_taxa = df_g.drop_duplicates("religiao").set_index("religiao").reindex(ordem).reset_index()
    fig.add_trace(go.Scatter(x=sub_taxa["religiao"], y=sub_taxa["taxa_diploma"],
                             name="% com diploma",
                             mode="lines+markers+text",
                             text=sub_taxa["taxa_diploma"].astype(str) + "%",
                             textposition="top center",
                             line=dict(color="#DC143C", width=3),
                             marker=dict(size=10)),
                  secondary_y=True)

    fig.update_yaxes(title_text="Renda mediana (salários mínimos)", secondary_y=False)
    fig.update_yaxes(title_text="% com ensino superior", secondary_y=True,
                     range=[0, max(sub_taxa["taxa_diploma"]) * 1.5])
    fig.update_layout(height=550, template="plotly_white", barmode="group",
                      title=f"Geração 1976-86 em 2010 — {genero}, {regiao}, {etnia}",
                      hovermode="x unified",
                      hoverlabel=dict(bgcolor="white", font_color="black"),
                      legend=dict(orientation="h", y=1.12))
    fig.show()

interact(grafico_berco,
         genero=Dropdown(options=["Todos", "Masculino", "Feminino"], description="Gênero:"),
         regiao=Dropdown(options=["Brasil", "Norte", "Nordeste", "Sudeste", "Sul", "Centro-Oeste"],
                         description="Região:"),
         etnia=Dropdown(options=["Todas", "Branca", "Negra (preta + parda)", "Preta", "Parda",
                                 "Amarela", "Indígena"], description="Etnia:"));

interactive(children=(Dropdown(description='Gênero:', options=('Todos', 'Masculino', 'Feminino'), value='Todos…